<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-geollms/blob/main/LLM_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 4.5 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd

# Step 1: Clone the GitHub repository if not already present
repo_url = "https://github.com/Dr-Isam-ALJAWARNEH/fds-project-geollms.git"
clone_dir = "fds-project-geollms"

if not os.path.exists(clone_dir):
    os.system(f"git clone {repo_url}")

# Step 2: Path to NDVI folder
ndvi_folder = os.path.join(clone_dir, "Datasets", "NDVI")

# Step 3: Read and analyze each CSV
all_dfs = []
print("Reading files from NDVI folder...\n")
for file in os.listdir(ndvi_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(ndvi_folder, file)
        df = pd.read_csv(file_path)
        all_dfs.append(df)
        print(f"File: {file}")
        print(f"    Rows: {len(df)}, Columns: {len(df.columns)}\n")

# Step 4: Combine all into one DataFrame
combined_ndvi_df = pd.concat(all_dfs, ignore_index=True)

# Step 5: Summary of combined data
print("Combined NDVI Dataset:")
print(f"    Total Rows: {len(combined_ndvi_df)}")
print(f"    Total Columns: {len(combined_ndvi_df.columns)}")
print("\n First 5 rows of the combined dataset:")
print(combined_ndvi_df.head())

# Check for missing values
missing_ndvi = combined_ndvi_df['grid_code'].isnull().sum()
missing_ndvi_coorx = combined_ndvi_df['x'].isnull().sum()
missing_ndvi_coory = combined_ndvi_df['y'].isnull().sum()
missing_datetime = combined_ndvi_df['Date'].isnull().sum()

# Display results
print(f"Missing values in 'grid_code': {missing_ndvi}")
print(f"Missing values in 'Date': {missing_datetime}")
print(f"Total number of rows after combining: {combined_ndvi_df.shape[0]}")
print(f"Missing values in 'x': {missing_ndvi_coorx}")
print(f"Missing values in 'y': {missing_ndvi_coory}")

# Print all column names
print("Columns in the combined dataset:")
print(combined_ndvi_df.columns.tolist())


Reading files from NDVI folder...

File: NDVI20221108.csv
    Rows: 1048575, Columns: 6

File: NDVI20220719.csv
    Rows: 1048575, Columns: 6

File: NDVI20220116.csv
    Rows: 1048575, Columns: 6

File: NDVI20230212.csv
    Rows: 1048575, Columns: 6

File: NDVI20211106.csv
    Rows: 1048575, Columns: 6

File: NDVI20221015.csv
    Rows: 1048575, Columns: 6

File: NDVI20220609.csv
    Rows: 1048575, Columns: 6

File: NDVI20211122.csv
    Rows: 1048575, Columns: 6

File: NDVI20230204.csv
    Rows: 1048575, Columns: 6

File: NDVI20220226.csv
    Rows: 1048575, Columns: 6

File: NDVI20220703.csv
    Rows: 1048575, Columns: 6

File: NDVI20221023.csv
    Rows: 1048575, Columns: 6

File: NDVI20210825.csv
    Rows: 1048575, Columns: 6

File: NDVI20220617.csv
    Rows: 1048575, Columns: 6

File: NDVI20210919.csv
    Rows: 1048575, Columns: 6

File: NDVI20210701.csv
    Rows: 1048575, Columns: 6

File: NDVI20221218.csv
    Rows: 1048575, Columns: 6

File: NDVI20220321.csv
    Rows: 1048575, Colum

In [2]:
import pandas as pd
import requests
from io import StringIO

# URL for GitHub files
base_url = "https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/AQ_data/"

# List of files to read
filenames = [f"chicago_eclipse_data_part_{i}.csv" for i in range(1, 20)]

# List to store DataFrames
dfs = []

# Download and read each CSV into a DataFrame
for filename in filenames:
    url = base_url + filename
    response = requests.get(url)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        dfs.append(df)
    else:
        print(f"Failed to load: {filename}")

# Combine all CSV files into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Check for missing values
missing_pm25 = combined_df['PM25'].isnull().sum()
missing_datetime = combined_df['ReadingDateTimeUTC'].isnull().sum()
missing_aq_coorx = combined_df['Latitude'].isnull().sum()
missing_aq_coory = combined_df['Longitude'].isnull().sum()

# Display results
print(f"Missing values in 'PM25': {missing_pm25}")
print(f"Missing values in 'ReadingDateTimeUTC': {missing_datetime}")
print(f"Total number of rows after combining: {combined_df.shape[0]}")
print(f"Missing values in 'Latitude': {missing_aq_coorx}")
print(f"Missing values in 'Longitude': {missing_aq_coory}")

# Print all column names
print("Columns in the combined dataset:")
print(combined_df.columns.tolist())


Missing values in 'PM25': 0
Missing values in 'ReadingDateTimeUTC': 0
Total number of rows after combining: 2461089
Missing values in 'Latitude': 0
Missing values in 'Longitude': 0
Columns in the combined dataset:
['City', 'DeviceId', 'LocationName', 'Latitude', 'Longitude', 'ReadingDateTimeUTC', 'PM25', 'CalibratedPM25', 'CalibratedO3', 'CalibratedNO2', 'CO', 'Temperature', 'Humidity', 'BatteryLevel', 'PercentBattery', 'CellSignal']


In [9]:
import json
from groq import Groq

# Initialize Groq client
client = Groq(api_key="YOUR_API")

# Define the LLM prompt for extraction
system_prompt = '''Extract origin, destination, and route preference from the user's travel request.
Valid preferences: "shortest", "healthiest", or "greenest" or combination of multiple of these eg. "shortest and healthiest", "shortest and healthiest and greenest". Handle typos and missing spaces if present, especially in locations names of Chicago City.
Please note that this system is designed for Chicago, Illinois, USA only and strictly. Note that healthiest related to air quality and greenest related to Scenic Vegetation.
Return ONLY the result in valid JSON format, with no additional text or explanation.
{
  "origin": "origin name",
  "destination": "destination name",
  "preference": "shortest | healthiest | greenest | healthiest and shortest | shortest and greenest | healthiest and greenest | shortest and healthiest and greenest "
}'''

# Define test prompts covering different combinations and typos
test_prompts = [
    "I want to go from Melrose Park to Hyde Park using the healthiest route",
    "Give me the greenest route from Lincoln Park to Garfield Park",
    "I need the shortest route between Loop and Hyde Park",
    "Please find the healthiest and greenest route from Melrose Park to Loop",
    "I'm going from Lincoln Park to Loop, I want the healthiest and shortest path",
    "Travel from Hyde Park to Garfield Park using the shortest and greenest route",
    "Melrose Park to Lincoln Park healthiest shortest greenest route please",
    "Go from HydePark to MelrosePark healthiest option",
    "How can I bike from Navy Pier to Hyde Park with the cleanest air?",
    "show me a route from Millennium Park to Logan Square avoiding polluted areas",
    "I want the most health-friendly route from Lincoln Park to Chinatown",
    "bike path from Bucktown to Loop with least pm25",
    "go from West Loop to Garfield Park with best air today",
    "What's the most scenic bike path from Wicker Park to the Museum Campus?",
    "Can I go from Uptown to South Shore through green areas?",
    "route from Roseland to Bronzeville with lots of trees",
    "i want to ride from Andersonville to downtown using the greenest way",
    "Bike me from Edgewater to Pilsen through leafy roads",
    "What's the shortest bike route from Loop to Lincoln Square?",
    "How quick is the ride from West Loop to Old Town?",
    "shortest biking path from South Side to North Side chicago",
    "fastest way from O'Hare to Logan Square by bike?",
    "Need shortest bike road from Humboldt park to Near North",
    "i wanna go frm lakeview to the field musuem green plz",
    "go from chinatown to uic campus w/ clean air n trees",
    "show me bike path from oak park to west loop (safe?)",
    "Best cycle way from McKinley park to the art institute healthy one plz",
    "hey i wanna bike frm rogers park to the loop — shortest & clean route",
    "wanna go biking from greektown to lincoln park with clean air and trees",
    "ride frm south loop to oakwood — prefer healthiest path pls",
    "i wanna go from logan sqr to the musuem campus – cleanest air pls",  # typo: "sqr" and "musuem"
    "bike route frm willies tower to grant pak, make it the shortest",   # typos: "willies" (Willis), "pak" (Park)
    "show me a path from south loop to boystown with trees n good air",  # typo-ish name: "boystown" (local nickname for Lakeview East)
    "frm brigdeport to linclon park — i want trees and clean air",       # typos: "brigdeport", "linclon"
    "give me a route from humblt park to sheed aquarim that’s green, shortest, and healthiest"  # typo: "humblt", "sheed aquarim", complex criteria
]

# Function to query the LLM
def extract_trip_info(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

# Run all test prompts and print results
for i, prompt in enumerate(test_prompts, start=1):
    print(f"\n--- Test Case {i} ---")
    print(f"Prompt: {prompt}")
    try:
        result = extract_trip_info(prompt)
        print("Extracted JSON:", json.dumps(result, indent=2))
    except Exception as e:
        print("Error parsing JSON:", str(e))



--- Test Case 1 ---
Prompt: I want to go from Melrose Park to Hyde Park using the healthiest route
Extracted JSON: {
  "origin": "Melrose Park",
  "destination": "Hyde Park",
  "preference": "healthiest"
}

--- Test Case 2 ---
Prompt: Give me the greenest route from Lincoln Park to Garfield Park
Extracted JSON: {
  "origin": "Lincoln Park",
  "destination": "Garfield Park",
  "preference": "greenest"
}

--- Test Case 3 ---
Prompt: I need the shortest route between Loop and Hyde Park
Extracted JSON: {
  "origin": "Loop",
  "destination": "Hyde Park",
  "preference": "shortest"
}

--- Test Case 4 ---
Prompt: Please find the healthiest and greenest route from Melrose Park to Loop
Extracted JSON: {
  "origin": "Melrose Park",
  "destination": "Loop",
  "preference": "healthiest and greenest"
}

--- Test Case 5 ---
Prompt: I'm going from Lincoln Park to Loop, I want the healthiest and shortest path
Extracted JSON: {
  "origin": "Lincoln Park",
  "destination": "Loop",
  "preference": "heal

In [4]:
import json
import pandas as pd
import networkx as nx
from scipy.spatial import cKDTree
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from math import sqrt
from groq import Groq

# -----------------------------
# Load Data
bike_url = 'https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/OSM%20datasets/chicago_bike_edges_useful_2.csv'
bike_df = pd.read_csv(bike_url, skipinitialspace=True)
aq_df = combined_df

# Step 3: Filter AQ and NDVI data to use only the latest reading per sensor location
aq_df['ReadingDateTimeUTC'] = pd.to_datetime(aq_df['ReadingDateTimeUTC'])
latest_aq_df = aq_df.sort_values('ReadingDateTimeUTC').groupby(['Latitude', 'Longitude']).tail(1)
latest_aq_coords = latest_aq_df[['Latitude', 'Longitude']].values
aq_tree = cKDTree(latest_aq_coords)
latest_ndvi_df = combined_ndvi_df.sort_values('Date').groupby(['x', 'y']).tail(1)
ndvi_coords = latest_ndvi_df[['x', 'y']].values
ndvi_tree = cKDTree(ndvi_coords)

# -----------------------------
# Graph Construction
G = nx.Graph()
for _, row in bike_df.iterrows():
    coords_str = row['geometry'].split('(')[1].split(')')[0]
    coords_list = [tuple(map(float, point.strip().split())) for point in coords_str.split(',')]
    start_coord = coords_list[0]
    end_coord = coords_list[-1]
    G.add_edge(start_coord, end_coord, weight=row['length'], geometry=coords_list)

# -----------------------------
# Utilities
geolocator = Nominatim(user_agent="geo_project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
location_overrides = {
    "melrose park": (-87.8567, 41.9006),
    "hyde park": (-87.5939, 41.7942),
    "loop": (-87.6278, 41.8819),
    "lincoln park": (-87.6376, 41.9214),
    "garfield park": (-87.7169, 41.8816)
}

def geocode_location(name):
    name_key = name.lower().strip()
    if name_key in location_overrides:
        return location_overrides[name_key]
    try_names = [ f"{name}, Chicago, Illinois, USA", f"{name}, Chicago, IL, USA", f"{name} Chicago IL", f"{name}, IL", f"{name}, Illinois",
        f"{name}, Cook County, IL", f"{name} near downtown Chicago", name]
    for query in try_names:
        location = geocode(query=query, exactly_one=True, bounded=False)
        if location:
            return (location.longitude, location.latitude)
    return None

def euclidean_distance(coord1, coord2):
    return sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

def yen_k_shortest_paths(graph, source, target, k, penalty_factor=2.0):
    paths = []
    temp_graph = graph.copy()
    for _ in range(k):
        path = nx.shortest_path(temp_graph, source, target, weight='cost')
        paths.append(path)
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            temp_graph[u][v]['cost'] *= penalty_factor
    return paths

def run_pathfinder(start_point, end_point, alpha=0.1, beta=0.05):
    local_G = G.copy()
    for u, v, data in local_G.edges(data=True):
        midpoint = ((u[0] + v[0]) / 2, (u[1] + v[1]) / 2)
        dist, idx = aq_tree.query([midpoint[1], midpoint[0]])
        pm25 = latest_aq_df.iloc[idx]['PM25']
        ndvi_dist, ndvi_idx = ndvi_tree.query([midpoint[0], midpoint[1]])
        ndvi_value = latest_ndvi_df.iloc[ndvi_idx]['grid_code']
        data['pm25'] = pm25
        data['ndvi'] = ndvi_value
        data['base_cost'] = data['weight'] * (1 + alpha * pm25 - beta * ndvi_value)
        data['cost'] = data['base_cost']

    nodes_list = list(local_G.nodes())
    bike_tree = cKDTree([(lat, lon) for lon, lat in nodes_list])
    _, start_idx = bike_tree.query((start_point[1], start_point[0]))
    _, end_idx = bike_tree.query((end_point[1], end_point[0]))
    start_node = nodes_list[start_idx]
    end_node = nodes_list[end_idx]

    local_G.add_edge(start_point, start_node, weight=euclidean_distance(start_point, start_node), pm25=0, ndvi=0, base_cost=0, cost=0)
    local_G.add_edge(end_point, end_node, weight=euclidean_distance(end_point, end_node), pm25=0, ndvi=0, base_cost=0, cost=0)

    top_k_paths = yen_k_shortest_paths(local_G, start_point, end_point, 1)
    best_path = top_k_paths[0]

    total_length_km = sum(local_G[u][v]['weight'] for u, v in zip(best_path[:-1], best_path[1:])) / 1000
    eta_minutes = (total_length_km / 15.0) * 60

    return best_path, total_length_km, eta_minutes

# -----------------------------
# LLM Setup
client = Groq(api_key="YOUR_API")
system_prompt = '''Extract origin, destination, and route preference from the user's travel request.
Valid preferences: "shortest", "healthiest", or "greenest" or combination of multiple of these eg. "shortest and healthiest", "shortest and healthiest and greenest". Handle typos and missing spaces if present, especially in locations names of Chicago City.
Please note that this system is designed for Chicago, Illinois, USA only and strictly. Note that healthiest related to air quality and greenest related to Scenic Vegetation.
Return ONLY the result in valid JSON format, with no additional text or explanation.
{
  "origin": "origin name",
  "destination": "destination name",
  "preference": "shortest | healthiest | greenest | healthiest and shortest | shortest and greenest | healthiest and greenest | shortest and healthiest and greenest "
}'''


def extract_trip_info(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    raw_output = response.choices[0].message.content.strip()
    print("=== RAW LLM OUTPUT ===")
    print(repr(raw_output))

    # Handle output wrapped in markdown code block
    if raw_output.startswith("```json"):
        raw_output = raw_output.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw_output)
    except json.JSONDecodeError as e:
        print("JSON decode failed:", str(e))
        return None

# -----------------------------
# Prompt Tests
test_prompts = [
    "I want to go from Melrose Park to Hyde Park using the healthiest route",
    "Give me the greenest route from Lincoln Park to Garfield Park",
    "I need the shortest route between Loop and Hyde Park",
    "Please find the healthiest and greenest route from Melrose Park to Loop",
    "I'm going from Lincoln Park to Loop, I want the healthiest and shortest path",
    "Travel from Hyde Park to Garfield Park using the shortest and greenest route",
    "Melrose Park to Lincoln Park healthiest shortest greenest route please",
    "Go from HydePark to MelrosePark healthiest option",
    "How can I bike from Navy Pier to Hyde Park with the cleanest air?",
    "show me a route from Millennium Park to Logan Square avoiding polluted areas",
    "I want the most health-friendly route from Lincoln Park to Chinatown",
    "bike path from Bucktown to Loop with least pm25",
    "go from West Loop to Garfield Park with best air today",
    "What's the most scenic bike path from Wicker Park to the Museum Campus?",
    "Can I go from Uptown to South Shore through green areas?",
    "route from Roseland to Bronzeville with lots of trees",
    "i want to ride from Andersonville to downtown using the greenest way",
    "Bike me from Edgewater to Pilsen through leafy roads",
    "What's the shortest bike route from Loop to Lincoln Square?",
    "How quick is the ride from West Loop to Old Town?",
    "shortest biking path from South Side to North Side chicago",
    "fastest way from O'Hare to Logan Square by bike?",
    "Need shortest bike road from Humboldt park to Near North",
    "i wanna go frm lakeview to the field musuem green plz",
    "go from chinatown to uic campus w/ clean air n trees",
    "show me bike path from oak park to west loop (safe?)",
    "Best cycle way from McKinley park to the art institute healthy one plz",
    "hey i wanna bike frm rogers park to the loop — shortest & clean route",
    "wanna go biking from greektown to lincoln park with clean air and trees",
    "ride frm south loop to oakwood — prefer healthiest path pls",
    "i wanna go from logan sqr to the musuem campus – cleanest air pls",  # typo: "sqr" and "musuem"
    "bike route frm willies tower to grant pak, make it the shortest",   # typos: "willies" (Willis), "pak" (Park)
    "show me a path from south loop to boystown with trees n good air",  # typo-ish name: "boystown" (local nickname for Lakeview East)
    "frm brigdeport to linclon park — i want trees and clean air",       # typos: "brigdeport", "linclon"
    "give me a route from humblt park to sheed aquarim that’s green, shortest, and healthiest"  # typo: "humblt", "sheed aquarim", complex criteria
]

# Mapping preference to weights
preference_to_weights = {
    "shortest": (0.0, 0.0),
    "healthiest": (0.2, 0.0),
    "greenest": (0.0, 0.2),
    "healthiest and greenest": (0.2, 0.2),
    "healthiest and shortest": (0.2, 0.0),
    "shortest and greenest": (0.0, 0.2),
    "shortest and healthiest and greenest": (0.2, 0.2),
}

# -----------------------------
# Run Test Cases
for i, prompt in enumerate(test_prompts, start=1):
    print(f"\n--- Test Case {i} ---")
    print(f"Prompt: {prompt}")
    parsed = extract_trip_info(prompt)
    if parsed is None:
        print("Skipping due to invalid LLM output.")
        continue

    origin = parsed['origin']
    destination = parsed['destination']
    preference = parsed['preference']
    alpha, beta = preference_to_weights.get(preference.lower(), (0.1, 0.05))

    start_coord = geocode_location(origin)
    end_coord = geocode_location(destination)
    if not start_coord or not end_coord:
        print("Geocoding failed for origin or destination.")
        continue

    path, dist, eta = run_pathfinder(start_coord, end_coord, alpha, beta)
    print(f"Route from {origin} to {destination}")
    print(f"   Preference: {preference} | Alpha: {alpha}, Beta: {beta}")
    print(f"   Distance: {dist:.2f} km | ETA: {eta:.1f} minutes")
    print(f"   Path (first 5): {path[:5]} ... total {len(path)} points")



--- Test Case 1 ---
Prompt: I want to go from Melrose Park to Hyde Park using the healthiest route
=== RAW LLM OUTPUT ===
'{\n  "origin": "Melrose Park",\n  "destination": "Hyde Park",\n  "preference": "healthiest"\n}'
Route from Melrose Park to Hyde Park
   Preference: healthiest | Alpha: 0.2, Beta: 0.0
   Distance: 33.34 km | ETA: 133.3 minutes
   Path (first 5): [(-87.8567, 41.9006), (-87.8489891, 41.9373179), (-87.8469649, 41.9373595), (-87.8448924, 41.9374018), (-87.8399585, 41.9374963)] ... total 536 points

--- Test Case 2 ---
Prompt: Give me the greenest route from Lincoln Park to Garfield Park
=== RAW LLM OUTPUT ===
'{\n  "origin": "Lincoln Park",\n  "destination": "Garfield Park",\n  "preference": "greenest"\n}'
Route from Lincoln Park to Garfield Park
   Preference: greenest | Alpha: 0.0, Beta: 0.2
   Distance: 10.41 km | ETA: 41.7 minutes
   Path (first 5): [(-87.6376, 41.9214), (-87.6376346, 41.9219726), (-87.6379964, 41.9219674), (-87.6382687, 41.9219634), (-87.6389413, 

Geocoding failed for origin or destination.

--- Test Case 26 ---
Prompt: show me bike path from oak park to west loop (safe?)
=== RAW LLM OUTPUT ===
'{\n  "origin": "Oak Park",\n  "destination": "West Loop",\n  "preference": "shortest"\n}'
Route from Oak Park to West Loop
   Preference: shortest | Alpha: 0.0, Beta: 0.0
   Distance: 13.16 km | ETA: 52.6 minutes
   Path (first 5): [(-87.794768, 41.8942018), (-87.7950502, 41.9090184), (-87.7948397, 41.9090221), (-87.7947848, 41.9090227), (-87.7944553, 41.9090281)] ... total 267 points

--- Test Case 27 ---
Prompt: Best cycle way from McKinley park to the art institute healthy one plz
=== RAW LLM OUTPUT ===
'{\n  "origin": "McKinley Park",\n  "destination": "The Art Institute of Chicago",\n  "preference": "healthiest"\n}'
Route from McKinley Park to The Art Institute of Chicago
   Preference: healthiest | Alpha: 0.2, Beta: 0.0
   Distance: 7.64 km | ETA: 30.6 minutes
   Path (first 5): [(-87.6736638, 41.8316997), (-87.6740906, 41.8319665)

Route from South Loop to Oakwood Hills
   Preference: healthiest | Alpha: 0.2, Beta: 0.0
   Distance: 30.65 km | ETA: 122.6 minutes
   Path (first 5): [(-87.623988, 41.865853), (-87.6241149, 41.8657947), (-87.6241357, 41.8666038), (-87.6241585, 41.8673984), (-87.6241574, 41.8675056)] ... total 456 points

--- Test Case 31 ---
Prompt: i wanna go from logan sqr to the musuem campus – cleanest air pls
=== RAW LLM OUTPUT ===
'{\n  "origin": "logan sqr",\n  "destination": "museum campus",\n  "preference": "healthiest"\n}'
Geocoding failed for origin or destination.

--- Test Case 32 ---
Prompt: bike route frm willies tower to grant pak, make it the shortest
=== RAW LLM OUTPUT ===
'{\n  "origin": "Willis Tower",\n  "destination": "Grant Park",\n  "preference": "shortest"\n}'
Route from Willis Tower to Grant Park
   Preference: shortest | Alpha: 0.0, Beta: 0.0
   Distance: 2.03 km | ETA: 8.1 minutes
   Path (first 5): [(-87.6359612, 41.878738), (-87.6365103, 41.8785242), (-87.6366529, 41.8785